# 3-round PerturbView decoding for SLIDE-0329 and SLIDE-0330

This version of the notebook updates the decoding workflow from **2 rounds / 8 bits** to **3 rounds / 12 bits**.

What changed:
- the decoder now works from a **CSV codebook** instead of a hard-coded `(R1_idx, R2_idx)` mapping
- decoding is now **round-agnostic**, so it handles `R1`, `R2`, and `R3`
- the slide loading / writing steps are set up for **SLIDE-0329** and **SLIDE-0330**
- the new R3 channels are included:
  - `R3_Bit9-RS0805-488`
  - `R3_Bit10-RS0763-Cy3B`
  - `R3_Bit11-RS1312-Cy5`
  - `R3_Bit12-RS0237-750`

Update the paths in the config cell if your Zarr or CSV locations differ.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spatialdata
import spatialdata as sd
from spatialdata import polygon_query

import scanpy as sc
import anndata as ad

ad.settings.allow_write_nullable_strings = True

try:
    import sopa
    sopa.settings.auto_save_on_disk = False
except Exception:
    sopa = None

from IPython.display import display


## Load the SpatialData objects

In [2]:
SLIDE_CONFIG = {
    "0329": {
        "zarr_path": r"C:\Analysis\M11_guidepool\SLIDE_0329.zarr",
        "table_key": "SLIDE_0329_CP_cells",
    },
    "0330": {
        "zarr_path": r"C:\Analysis\M11_guidepool\SLIDE_0330.zarr",
        "table_key": "SLIDE_0330_CP_cells",
    },
}

sdata_by_slide = {}

for slide_id, cfg in SLIDE_CONFIG.items():
    sdata_by_slide[slide_id] = spatialdata.read_zarr(cfg["zarr_path"])
    print(f"Loaded SLIDE-{slide_id}:")
    print(sdata_by_slide[slide_id])
    print()

version mismatch: detected: RasterFormatV02, requested: FormatV04


Loaded SLIDE-0329:


version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: C:\Analysis\M11_guidepool\SLIDE_0329.zarr
├── Images
│     └── 'SLIDE-0329': DataTree[cyx] (15, 62617, 66406), (15, 31308, 33203), (15, 15654, 16601), (15, 7827, 8300), (15, 3913, 4150), (15, 1956, 2075), (15, 978, 1037), (15, 489, 518), (15, 244, 259), (15, 122, 129)
├── Shapes
│     ├── 'all_tumors': GeoDataFrame shape: (5, 1) (2D shapes)
│     ├── 'cp_DAPI_f04_p0_s01': GeoDataFrame shape: (1853806, 1) (2D shapes)
│     ├── 'image_patches': GeoDataFrame shape: (233, 3) (2D shapes)
│     ├── 'tumor_B5_A_1L': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'tumor_B5_A_1R': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'tumor_B5_A_1R1L': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'tumor_B5_A_2R': GeoDataFrame shape: (1, 1) (2D shapes)
│     └── 'tumor_B5_A_NH': GeoDataFrame shape: (2, 1) (2D shapes)
└── Tables
      └── 'SLIDE_0329_CP_cells': AnnData (1853806, 15)
with coordinate systems:
    ▸ 'SLIDE-0329', with elements:
      

## Add tumor IDs from tumor shape elements

In [3]:
from shapely.ops import unary_union


def _normalize_name(x: str) -> str:
    return str(x).replace("-", "_").lower()


def infer_table_key(sdata, preferred: str | None = None) -> str:
    table_keys = list(sdata.tables.keys())

    if preferred is not None and preferred in table_keys:
        return preferred

    if preferred is not None:
        preferred_norm = _normalize_name(preferred)
        for k in table_keys:
            if _normalize_name(k) == preferred_norm:
                return k

    if len(table_keys) == 1:
        return table_keys[0]

    cp_like = [k for k in table_keys if _normalize_name(k).endswith("_cp_cells")]
    if len(cp_like) == 1:
        return cp_like[0]

    raise ValueError(
        f"Could not infer table key. Available tables: {table_keys}. "
        "Set SLIDE_CONFIG[slide_id]['table_key'] explicitly."
    )


def infer_coordinate_system(sdata, preferred: str | None = "global") -> str:
    coords = [str(c) for c in sdata.coordinate_systems]

    if preferred is not None and preferred in coords:
        return preferred

    non_global = [c for c in coords if c != "global"]
    if len(non_global) == 1:
        return non_global[0]

    if len(coords) == 1:
        return coords[0]

    raise ValueError(
        f"Could not infer coordinate system. Available coordinate systems: {coords}. "
        "Set the coord argument explicitly."
    )


def add_tumor_id_from_shapes(
    sdata,
    table_name: str | None = None,
    coord: str | None = "global",
    shape_prefix: str = "tumor_",
    tumor_id_col: str = "tumor_id",
    unassigned_label: str = "unassigned",
) -> dict[str, pd.Index]:
    """
    Add or overwrite `tumor_id_col` in `sdata[table_name].obs` by polygon-querying all
    shape elements whose names start with `shape_prefix`.

    This version is robust to:
      - non-global coordinate systems (e.g. only 'SLIDE-0329')
      - table key naming differences (hyphen vs underscore)
      - tumor shape elements that contain multiple polygons
      - empty polygon queries for a given tumor
    """
    table_name = infer_table_key(sdata, table_name)
    coord = infer_coordinate_system(sdata, coord)
    table = sdata[table_name]

    if not table.obs.index.is_unique:
        raise ValueError(f"{table_name}.obs.index must be unique")

    tumors = sorted([k for k in sdata.shapes.keys() if k.startswith(shape_prefix)])
    categories = [unassigned_label] + tumors

    table.obs[tumor_id_col] = pd.Categorical(
        [unassigned_label] * table.n_obs,
        categories=categories,
    )

    tumor_to_index: dict[str, pd.Index] = {}
    for tumor in tumors:
        geoms = sdata[tumor].geometry
        poly = geoms.iloc[0] if len(geoms) == 1 else unary_union(list(geoms))

        try:
            queried = polygon_query(
                sdata,
                polygon=poly,
                target_coordinate_system=coord,
            )
        except AssertionError:
            tumor_to_index[tumor] = pd.Index([], dtype=object)
            continue

        if table_name in queried.tables:
            idx = queried[table_name].obs.index
        else:
            idx = pd.Index([], dtype=object)

        if len(idx) > 0:
            table.obs.loc[idx, tumor_id_col] = tumor

        tumor_to_index[tumor] = idx

    return tumor_to_index

In [4]:
for slide_id, cfg in SLIDE_CONFIG.items():
    cfg["table_key"] = infer_table_key(sdata_by_slide[slide_id], cfg.get("table_key"))
    cfg["coord"] = infer_coordinate_system(sdata_by_slide[slide_id], cfg.get("coord", "global"))

    print(
        f"SLIDE-{slide_id}: using table_key={cfg['table_key']!r}, "
        f"coord={cfg['coord']!r}"
    )

    _ = add_tumor_id_from_shapes(
        sdata_by_slide[slide_id],
        table_name=cfg["table_key"],
        coord=cfg["coord"],
    )

for slide_id, cfg in SLIDE_CONFIG.items():
    print(f"SLIDE-{slide_id} tumor counts:")
    print(sdata_by_slide[slide_id][cfg["table_key"]].obs["tumor_id"].value_counts(dropna=False))
    print()

SLIDE-0329: using table_key='SLIDE_0329_CP_cells', coord='SLIDE-0329'
SLIDE-0330: using table_key='SLIDE_0330_CP_cells', coord='SLIDE-0330'
SLIDE-0329 tumor counts:
tumor_id
tumor_B5_A_NH      589850
tumor_B5_A_1R1L    409887
tumor_B5_A_1L      312165
tumor_B5_A_1R      279598
tumor_B5_A_2R      262306
unassigned              0
Name: count, dtype: int64

SLIDE-0330 tumor counts:
tumor_id
tumor_C2_A_2R      331152
tumor_C2_A_1R1L    292770
tumor_C2_A_1L      270096
tumor_C2_A_1R      251533
tumor_C2_A_NH      206916
unassigned              0
Name: count, dtype: int64



## Decoding helpers

In [5]:
EPS = 1e-9


def percentile_scale(df, cols, p=95, target=1.0):
    out = df[cols].copy()
    for c in cols:
        pval = np.percentile(df[c].values, p)
        scale = pval if pval > 0 else 1.0
        out[c] = df[c] / (scale + EPS) * target
    return out


def call_round(norm_block: pd.DataFrame):
    X = norm_block.values
    top_idx = X.argmax(axis=1)
    top_val = X.max(axis=1)
    second = np.partition(X, -2, axis=1)[:, -2]
    ratio = top_val / (second + EPS)
    return dict(top_idx=top_idx, top_val=top_val, ratio=ratio)


def null_thresholds_nonwinners(raw_block: pd.DataFrame, top_idx: np.ndarray, null_quant=99.0):
    X = raw_block.values
    cols = list(raw_block.columns)
    thresholds = {}
    for k, c in enumerate(cols):
        null_vals = X[top_idx != k, k]
        thr = np.percentile(null_vals, null_quant) if null_vals.size else np.percentile(X[:, k], null_quant)
        thresholds[c] = float(thr)
    return thresholds


def winner_raw_top(df, cols, top_idx):
    X = df[cols].to_numpy()
    idx = top_idx.astype(int)
    return X[np.arange(len(df)), idx]


def ratio_quality(ratio, k=0.6, anchor=1.0):
    """
    Smoothly map the top/second ratio to [0, 1].
    """
    return np.clip(1.0 - np.exp(-k * (ratio - anchor)), 0.0, 1.0)


def pass_funnel(df, round_prefix):
    top = df[f"{round_prefix}_pass_top"]
    metr = df[f"{round_prefix}_pass_metric"]
    total = len(df)
    top_pass = int(top.sum())
    both_pass = int((top & metr).sum())
    return pd.Series({
        "total_cells": total,
        "pass_top": top_pass,
        "lost_at_top": total - top_pass,
        "pass_metric_given_top": both_pass,
        "lost_at_metric_after_top": top_pass - both_pass,
        "final_round_pass": both_pass,
    })


def tuple_to_bitstring(idx_tuple, bits_per_round=4):
    chunks = []
    for idx in idx_tuple:
        chunk = ["0"] * bits_per_round
        if idx is not None and idx >= 0:
            chunk[int(idx)] = "1"
        chunks.append("".join(chunk))
    return "".join(chunks)


def build_codebook_from_csv(
    csv_path,
    guide_col="base",
    bits_col="bits",
    round_names=("R1", "R2", "R3"),
    bits_per_round=4,
):
    """
    Read a CSV with one code string per guide and convert it to:
    - a dataframe with parsed round tuples
    - a dict mapping (R1_idx, R2_idx, R3_idx, ...) -> guide name
    """
    cb = pd.read_csv(csv_path, dtype={bits_col: str}).copy()
    n_bits = len(round_names) * bits_per_round

    cb[bits_col] = (
        cb[bits_col]
        .astype(str)
        .str.replace(r"\s+", "", regex=True)
        .str.zfill(n_bits)
    )

    def parse_bitstring(bitstr):
        if len(bitstr) != n_bits:
            raise ValueError(f"Expected {n_bits} bits, got {len(bitstr)} for {bitstr}")
        idxs = []
        for i, round_name in enumerate(round_names):
            chunk = bitstr[i * bits_per_round:(i + 1) * bits_per_round]
            on = [j for j, b in enumerate(chunk) if b == "1"]
            if len(on) != 1:
                raise ValueError(
                    f"Guide code {bitstr} has invalid {round_name} chunk {chunk}; "
                    "expected exactly one '1' per round."
                )
            idxs.append(on[0])
        return tuple(idxs)

    cb["round_tuple"] = cb[bits_col].map(parse_bitstring)
    cb["round_label"] = cb["round_tuple"].map(
        lambda t: "_".join(f"{r}C{idx + 1}" for r, idx in zip(round_names, t))
    )

    if cb["round_tuple"].duplicated().any():
        dup = cb.loc[cb["round_tuple"].duplicated(keep=False), [guide_col, bits_col, "round_tuple"]]
        raise ValueError(f"Duplicate round tuples found in codebook:\n{dup}")

    codebook = dict(zip(cb["round_tuple"], cb[guide_col]))
    return cb, codebook


def decode_perturbview_signals(
    sdata,
    table_key: str,
    round_cols: dict[str, list[str]],
    codebook: dict[tuple, str],
    *,
    UNKNOWN_LABEL: str = "UNK",
    RATIO_MIN: float = 2.0,
    NULL_QUANT: float = 95.0,
    NORM_P: float = 99.99,
    bits_per_round: int = 4,
):
    """
    Decode each cell across an arbitrary number of rounds.

    Returns
    -------
    df : pd.DataFrame
        obs + expression + decode columns
    funnel : pd.DataFrame
        per-round pass statistics
    summary_calls : pd.DataFrame
        counts of final guide calls
    thresholds_by_round : dict
        raw non-winner thresholds for each round/channel
    """
    adata = sdata.tables[table_key]
    df = adata.obs.join(pd.DataFrame(adata.to_df(), index=adata.obs_names))

    round_names = list(round_cols.keys())
    thresholds_by_round = {}

    for round_name, cols in round_cols.items():
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise KeyError(f"{table_key} is missing columns for {round_name}: {missing}")

        norm = percentile_scale(df, cols, p=NORM_P, target=1.0)
        call = call_round(norm)

        df[f"{round_name}_top_idx"] = call["top_idx"]
        df[f"{round_name}_ratio"] = call["ratio"]

        thr_raw = null_thresholds_nonwinners(
            df[cols],
            df[f"{round_name}_top_idx"].values,
            null_quant=NULL_QUANT,
        )
        thresholds_by_round[round_name] = thr_raw

        df[f"{round_name}_raw_top"] = winner_raw_top(df, cols, df[f"{round_name}_top_idx"].values)
        df[f"{round_name}_top_thr"] = np.array([thr_raw[c] for c in cols])[df[f"{round_name}_top_idx"].values]
        df[f"{round_name}_top_fold"] = df[f"{round_name}_raw_top"] / (df[f"{round_name}_top_thr"] + EPS)

        df[f"{round_name}_pass_top"] = df[f"{round_name}_raw_top"] >= df[f"{round_name}_top_thr"]
        df[f"{round_name}_pass_metric"] = df[f"{round_name}_ratio"] >= RATIO_MIN
        df[f"{round_name}_pass"] = df[f"{round_name}_pass_top"] & df[f"{round_name}_pass_metric"]
        df[f"{round_name}_quality"] = ratio_quality(df[f"{round_name}_ratio"])

    round_tuples = list(zip(*[df[f"{r}_top_idx"] for r in round_names]))
    df["round_label"] = [
        "_".join(f"{r}C{idx + 1}" for r, idx in zip(round_names, t))
        for t in round_tuples
    ]
    df["decoded_bits"] = [tuple_to_bitstring(t, bits_per_round=bits_per_round) for t in round_tuples]

    mapped_guides = np.array([codebook.get(t, UNKNOWN_LABEL) for t in round_tuples], dtype=object)
    all_pass = np.logical_and.reduce([df[f"{r}_pass"].to_numpy() for r in round_names])
    df["guide_call"] = np.where(all_pass, mapped_guides, "None")

    qual_stack = np.vstack([df[f"{r}_quality"].to_numpy() for r in round_names])
    geom_mean_quality = np.exp(np.mean(np.log(np.clip(qual_stack, 1e-12, 1.0)), axis=0))
    df["call_confidence"] = np.where(df["guide_call"] != "None", geom_mean_quality, 0.0)

    funnel = pd.DataFrame({r: pass_funnel(df, r) for r in round_names})

    summary_calls = (
        df["guide_call"]
        .value_counts(dropna=False)
        .rename_axis("guide")
        .reset_index(name="n_cells")
        .sort_values("n_cells", ascending=False)
    )

    return df, funnel, summary_calls, thresholds_by_round


def update_spatialdata_obs_with_decode(
    sdata,
    table_key: str,
    df: pd.DataFrame,
    *,
    round_names: list[str] | None = None,
    prefix: str = "decode_",
    cols_to_copy: list[str] | None = None,
    add_guide_alias: bool = True,
) -> None:
    """
    Copy decode / QC columns from `df` into `sdata.tables[table_key].obs` in-place.
    Safe to re-run: existing prefixed decode columns are dropped before re-joining.
    """
    adata = sdata.tables[table_key]

    if round_names is None:
        round_names = sorted({c.split("_")[0] for c in df.columns if c.startswith("R") and "_" in c})

    if cols_to_copy is None:
        cols_to_copy = []
        for r in round_names:
            cols_to_copy.extend([
                f"{r}_top_idx",
                f"{r}_ratio",
                f"{r}_raw_top",
                f"{r}_top_thr",
                f"{r}_top_fold",
                f"{r}_pass_top",
                f"{r}_pass_metric",
                f"{r}_pass",
                f"{r}_quality",
            ])
        cols_to_copy.extend([
            "round_label",
            "decoded_bits",
            "guide_call",
            "call_confidence",
        ])

    avail = [c for c in cols_to_copy if c in df.columns]
    df_sub = df[avail].copy().reindex(adata.obs_names)

    bool_cols = {c for c in df_sub.columns if c.endswith(("_pass_top", "_pass_metric", "_pass"))}
    int_cols = {c for c in df_sub.columns if c.endswith("_top_idx")}
    str_cols = {"round_label", "decoded_bits", "guide_call"}

    for c in bool_cols & set(df_sub.columns):
        df_sub[c] = df_sub[c].fillna(False).astype(bool)
    for c in int_cols & set(df_sub.columns):
        df_sub[c] = df_sub[c].fillna(-1).astype("Int32")
    for c in str_cols & set(df_sub.columns):
        df_sub[c] = df_sub[c].fillna("None").astype("string")

    prefixed_cols = [prefix + c for c in df_sub.columns]
    existing = [c for c in prefixed_cols if c in adata.obs.columns]
    if existing:
        adata.obs = adata.obs.drop(columns=existing)

    adata.obs = adata.obs.join(df_sub.add_prefix(prefix), how="left")

    if add_guide_alias and (prefix + "guide_call") in adata.obs.columns:
        if "guide" in adata.obs.columns:
            adata.obs = adata.obs.drop(columns=["guide"])
        adata.obs["guide"] = adata.obs[prefix + "guide_call"].astype("category")


## Configure the 3 decoding rounds and build the codebook from CSV

In [9]:
ROUND_COLS = {
    "R1": [
        "R1_Bit1-RS0996-488",
        "R1_Bit2-RS0584-Cy3B",
        "R1_Bit3-RS0015-Cy5",
        "R1_Bit4-RS0083-750",
    ],
    "R2": [
        "R2_Bit5-RS1047-488",
        "R2_Bit6-RS0639-Cy3B",
        "R2_Bit7-RS0109-Cy5",
        "R2_Bit8-RS0255-750",
    ],
    "R3": [
        "R3_Bit9-RS0805-488",
        "R3_Bit10-RS0763-Cy3B",
        "R3_Bit11-RS1312-Cy5",
        "R3_Bit12-RS0237-750",
    ],
}

CODEBOOK_CSV = Path("BL1_2_targets.csv")

codebook_df, codebook = build_codebook_from_csv(
    CODEBOOK_CSV,
    guide_col="base",
    bits_col="bits",
    round_names=list(ROUND_COLS.keys()),
    bits_per_round=4,
)

display(codebook_df[["base", "bits", "round_tuple", "round_label"]].head(20))
print(f"{len(codebook)} guide codes loaded from {CODEBOOK_CSV}")


,base,bits,round_tuple,round_label
0,BL1.1,100010001000,"(0, 0, 0)",R1C1_R2C1_R3C1
1,BL1.2,010010001000,"(1, 0, 0)",R1C2_R2C1_R3C1
2,BL1.3,001010001000,"(2, 0, 0)",R1C3_R2C1_R3C1
3,BL1.4,000110001000,"(3, 0, 0)",R1C4_R2C1_R3C1
4,BL1.5,100001001000,"(0, 1, 0)",R1C1_R2C2_R3C1
5,BL1.6,010001001000,"(1, 1, 0)",R1C2_R2C2_R3C1
6,BL1.7,001001001000,"(2, 1, 0)",R1C3_R2C2_R3C1
7,BL1.8,000101001000,"(3, 1, 0)",R1C4_R2C2_R3C1
8,BL1.9,100000101000,"(0, 2, 0)",R1C1_R2C3_R3C1
9,BL1.10,010000101000,"(1, 2, 0)",R1C2_R2C3_R3C1


40 guide codes loaded from BL1_2_targets.csv


In [10]:
for slide_id, cfg in SLIDE_CONFIG.items():
    adata = sdata_by_slide[slide_id][cfg["table_key"]]
    var_names = set(map(str, adata.var_names))
    missing = {
        round_name: [c for c in cols if c not in var_names]
        for round_name, cols in ROUND_COLS.items()
    }
    missing = {k: v for k, v in missing.items() if v}
    print(f"SLIDE-{slide_id}")
    if missing:
        print("  Missing columns:")
        for k, v in missing.items():
            print(f"    {k}: {v}")
    else:
        print("  All decode columns found.")
    print()


SLIDE-0329
  All decode columns found.

SLIDE-0330
  All decode columns found.



## Decode both slides

In [11]:
decode_results = {}

for slide_id, cfg in SLIDE_CONFIG.items():
    print(f"===== Decoding SLIDE-{slide_id} =====")

    df, funnel, summary_calls, thresholds_by_round = decode_perturbview_signals(
        sdata_by_slide[slide_id],
        table_key=cfg["table_key"],
        round_cols=ROUND_COLS,
        codebook=codebook,
    )

    decode_results[slide_id] = {
        "df": df,
        "funnel": funnel,
        "summary_calls": summary_calls,
        "thresholds_by_round": thresholds_by_round,
    }

    for round_name, thr in thresholds_by_round.items():
        print(f"{round_name} thresholds:")
        print(thr)

    print()
    print("Pass funnel:")
    display(funnel)

    print("Top guide calls:")
    display(summary_calls.head(20))

    update_spatialdata_obs_with_decode(
        sdata_by_slide[slide_id],
        table_key=cfg["table_key"],
        df=df,
        round_names=list(ROUND_COLS.keys()),
        prefix="decode_",
    )


===== Decoding SLIDE-0329 =====
R1 thresholds:
{'R1_Bit1-RS0996-488': 795.0189573459716, 'R1_Bit2-RS0584-Cy3B': 2052.924376861687, 'R1_Bit3-RS0015-Cy5': 1395.6682500588124, 'R1_Bit4-RS0083-750': 1354.2015973854966}
R2 thresholds:
{'R2_Bit5-RS1047-488': 628.5104881874485, 'R2_Bit6-RS0639-Cy3B': 775.3931463222453, 'R2_Bit7-RS0109-Cy5': 945.635103926097, 'R2_Bit8-RS0255-750': 401.04200840999994}
R3 thresholds:
{'R3_Bit9-RS0805-488': 3921.184968449765, 'R3_Bit10-RS0763-Cy3B': 3012.4793580114383, 'R3_Bit11-RS1312-Cy5': 3280.808761203583, 'R3_Bit12-RS0237-750': 80.6498485295685}

Pass funnel:


,R1,R2,R3
total_cells,1853806,1853806,1853806
pass_top,977391,978185,17159
lost_at_top,876415,875621,1836647
pass_metric_given_top,809197,798323,12239
lost_at_metric_after_top,168194,179862,4920
final_round_pass,809197,798323,12239


Top guide calls:


,guide,n_cells
0,None,1848265
1,UNK,5540
2,BL1.3,1


===== Decoding SLIDE-0330 =====
R1 thresholds:
{'R1_Bit1-RS0996-488': 742.399288915339, 'R1_Bit2-RS0584-Cy3B': 1677.774606686332, 'R1_Bit3-RS0015-Cy5': 1485.2348321947786, 'R1_Bit4-RS0083-750': 1338.9003021148037}
R2 thresholds:
{'R2_Bit5-RS1047-488': 668.0430908424792, 'R2_Bit6-RS0639-Cy3B': 867.0145083795544, 'R2_Bit7-RS0109-Cy5': 943.09366562553, 'R2_Bit8-RS0255-750': 367.5381360636064}
R3 thresholds:
{'R3_Bit9-RS0805-488': 4976.610843161417, 'R3_Bit10-RS0763-Cy3B': 3845.9486496046547, 'R3_Bit11-RS1312-Cy5': 4788.276421837176, 'R3_Bit12-RS0237-750': 80.4044484184643}

Pass funnel:


,R1,R2,R3
total_cells,1352467,1352467,1352467
pass_top,783632,764600,18551
lost_at_top,568835,587867,1333916
pass_metric_given_top,678383,649491,12997
lost_at_metric_after_top,105249,115109,5554
final_round_pass,678383,649491,12997


Top guide calls:


,guide,n_cells
0,None,1345074
1,UNK,7391
2,BL1.19,1
3,BL1.3,1


In [12]:
for slide_id, cfg in SLIDE_CONFIG.items():
    print(f"SLIDE-{slide_id}")
    obs = sdata_by_slide[slide_id][cfg["table_key"]].obs
    display(
        obs[
            [
                "tumor_id",
                "decode_round_label",
                "decode_decoded_bits",
                "decode_guide_call",
                "decode_call_confidence",
            ]
        ].head()
    )


SLIDE-0329


,tumor_id,decode_round_label,decode_decoded_bits,decode_guide_call,decode_call_confidence
aaaaaaaa-1,tumor_B5_A_2R,R1C2_R2C1_R3C4,010010000001,None,0.0
aaaaaaab-1,tumor_B5_A_2R,R1C1_R2C1_R3C4,100010000001,None,0.0
aaaaaaac-1,tumor_B5_A_2R,R1C2_R2C4_R3C4,010000010001,None,0.0
aaaaaaad-1,tumor_B5_A_2R,R1C2_R2C4_R3C4,010000010001,None,0.0
aaaaaaae-1,tumor_B5_A_2R,R1C2_R2C4_R3C4,010000010001,None,0.0


SLIDE-0330


,tumor_id,decode_round_label,decode_decoded_bits,decode_guide_call,decode_call_confidence
aaaaaaaa-1,tumor_C2_A_2R,R1C2_R2C2_R3C4,010001000001,None,0.0
aaaaaaab-1,tumor_C2_A_2R,R1C1_R2C1_R3C4,100010000001,None,0.0
aaaaaaac-1,tumor_C2_A_2R,R1C3_R2C4_R3C4,001000010001,None,0.0
aaaaaaad-1,tumor_C2_A_2R,R1C1_R2C1_R3C4,100010000001,None,0.0
aaaaaaae-1,tumor_C2_A_2R,R1C3_R2C4_R3C4,001000010001,None,0.0


## Write decoded tables back to the SpatialData objects

In [13]:
sdata_by_slide

{'0329': SpatialData object, with associated Zarr store: C:\Analysis\M11_guidepool\SLIDE_0329.zarr
 ├── Images
 │     └── 'SLIDE-0329': DataTree[cyx] (15, 62617, 66406), (15, 31308, 33203), (15, 15654, 16601), (15, 7827, 8300), (15, 3913, 4150), (15, 1956, 2075), (15, 978, 1037), (15, 489, 518), (15, 244, 259), (15, 122, 129)
 ├── Shapes
 │     ├── 'all_tumors': GeoDataFrame shape: (5, 1) (2D shapes)
 │     ├── 'cp_DAPI_f04_p0_s01': GeoDataFrame shape: (1853806, 1) (2D shapes)
 │     ├── 'image_patches': GeoDataFrame shape: (233, 3) (2D shapes)
 │     ├── 'tumor_B5_A_1L': GeoDataFrame shape: (1, 2) (2D shapes)
 │     ├── 'tumor_B5_A_1R': GeoDataFrame shape: (1, 2) (2D shapes)
 │     ├── 'tumor_B5_A_1R1L': GeoDataFrame shape: (1, 2) (2D shapes)
 │     ├── 'tumor_B5_A_2R': GeoDataFrame shape: (1, 2) (2D shapes)
 │     └── 'tumor_B5_A_NH': GeoDataFrame shape: (2, 1) (2D shapes)
 └── Tables
       └── 'SLIDE_0329_CP_cells': AnnData (1853806, 15)
 with coordinate systems:
     ▸ 'SLIDE-0329

In [ ]:
for slide_id, cfg in SLIDE_CONFIG.items():
    sdata = sdata_by_slide[slide_id]
    src_table = cfg["table_key"]
    decoded_table = f"{src_table}_decoded"

    sdata.tables[decoded_table] = sdata.tables[src_table]
    sdata.write_element(decoded_table, overwrite=True)

    print(f"Wrote {decoded_table}")


## Optional: export decoded tables to h5ad

In [ ]:
for slide_id, cfg in SLIDE_CONFIG.items():
    decoded_table = f"{cfg['table_key']}_decoded"
    out_path = f"{decoded_table}.h5ad"
    sdata_by_slide[slide_id][decoded_table].write_h5ad(out_path)
    print(f"Wrote {out_path}")
